# RNA-Seq Processing Pipeline
### Authors: Emily Skates, Stephen Williams

Processes STAR + featureCounts outputs through downstream analyses including DESeq2 differential expression, PCA/PERMANOVA clustering, WGCNA co-expression networks, and Gene Ontology/GSEA enrichment.

## 1. Environment Setup & Global Configuration
Initialises required libraries, sets the working directory, and defines global thresholds for gene filtering, plotting, and differential expression cut-offs.

In [ ]:
setwd("/Users/steve/Library/CloudStorage/GoogleDrive-stev3.w1l@gmail.com/My Drive/Academia/c_collaborations/emilyS/")

required_packages <- c("DESeq2", "ggplot2", "ggvenn", "WGCNA", "vegan", "dplyr", "tidyr", "tibble",
                       "pairwiseAdonis", "glue", "scales", "gtools", "clusterProfiler",
                       "enrichplot", "org.Mm.eg.db", "patchwork")

missing_packages <- required_packages[!(required_packages %in% installed.packages()[, "Package"])]
if (length(missing_packages)) {
    install.packages(missing_packages)
}

suppressWarnings(
    suppressPackageStartupMessages({
        invisible(lapply(required_packages, library, character.only = TRUE))
    })
)

min_gene_copies           <- 10
min_samples_per_condition <- 1
num_pca_genes             <- 500
num_network_genes         <- 100
log2_fc_threshold         <- 0.5
adjusted_p_threshold      <- 0.05
nominal_p_threshold       <- 0.05

options(repr.plot.width = 8, repr.plot.height = 4)

## 2. Data Loading & DESeq2 Initialisation
Compiles the individual gene count files into a unified matrix, defines the sample metadata (including the nested subject design), and initialises the `DESeqDataSet`.

### Understanding the Experimental Design Formula

The DESeq2 design formula breaks down the total variance in gene expression into specific biological, experimental, and technical components:

$$\text{Design Matrix} \sim \text{LabelingState} + \text{LabelingState:SubjectId} + \text{AssayType} + \text{LabelingState:AssayType}$$

* **`LabelingState`**: Controls for the baseline difference between the labeled and unlabeled conditions (e.g., global transcriptomic shifts caused by the metabolic labeling itself).
* **`LabelingState:SubjectId`**: Accounts for individual biological variation by nesting the samples strictly within their respective labeling groups. This captures the paired relationship between an individual sample's Input and Pull-Down samples, preventing sample-to-sample baseline differences from muddying the results.
* **`AssayType`**: Measures the non-specific background enrichment. It captures the genes that naturally stick to the beads or tube during a pull-down even when no label is present:
  $$\text{Unlabeled Pull-Down} - \text{Unlabeled Input}$$
* **`LabelingState:AssayType`**: The interaction term between label/pulldown state that isolates the **True Enrichment**. It calculates the "difference of differences," subtracting non-specific background stickiness from the labeled enrichment to find genes specifically captured by the assay:

$$\text{True Target Enrichment} = (\text{Labeled Pull-Down} - \text{Labeled Input}) - (\text{Unlabeled Pull-Down} - \text{Unlabeled Input})$$

In [ ]:
count_files <- mixedsort(Sys.glob("o_outputs/sample_*/gene_counts.txt"))

initial_file_data <- read.table(count_files[1], header = TRUE, skip = 1, stringsAsFactors = FALSE)
unified_count_matrix <- data.frame(Geneid = initial_file_data[, 1])

for (file_path in count_files) {
    sample_id <- basename(dirname(file_path))
    sample_data <- read.table(file_path, header = TRUE, skip = 1, stringsAsFactors = FALSE)
    unified_count_matrix[[sample_id]] <- sample_data[, 7]
}

row.names(unified_count_matrix) <- unified_count_matrix$Geneid
unified_count_matrix$Geneid <- NULL

experimental_metadata <- data.frame(
    LabelingState = factor(rep(c("unlab", "lab", "unlab", "lab"), each = 3), levels = c("unlab", "lab")),
    AssayType = factor(rep(c("pull_down", "input"), each = 6), levels = c("input", "pull_down")),
    SubjectId = factor(rep(c("Subj1", "Subj2", "Subj3"), times = 4))
)
row.names(experimental_metadata) <- colnames(unified_count_matrix)

deseq_dataset <- DESeqDataSetFromMatrix(
    countData = unified_count_matrix,
    colData = experimental_metadata,
    design = ~ LabelingState + LabelingState:SubjectId + AssayType + LabelingState:AssayType
)

genes_to_keep <- rowSums(counts(deseq_dataset) >= min_gene_copies) >= min_samples_per_condition
deseq_dataset <- deseq_dataset[genes_to_keep, ]
vst_transformed_data <- vst(deseq_dataset, blind = FALSE)

## 3. Dispersion Diagnostics
Calculates size factors and dispersion estimates, plotting the fit to ensure data quality and model appropriateness.

### Understanding DESeq2 Dispersion Plots

In RNA-Seq data, gene expression variance naturally scales with the mean (highly expressed genes show higher absolute variance). 
To model this accurately, DESeq2 estimates a **dispersion** parameter for each gene, which measures how much a gene's variance exceeds its mean.

A standard diagnostics plot visualises this relationship across three layers:
1. **Black Dots (`dispGeneEst`)**: The raw, independent dispersion estimate for each individual gene based purely on its sample variance.
2. **Red Line (`dispFit`)**: The global trend line fit across all genes, capturing how dispersion typically decreases as the mean expression increases.
3. **Blue Dots (`dispersion`)**: The final, shrunken dispersion values used for differential expression testing. DESeq2 shrinks gene-wise estimates toward the red line to minimise false positives from low-abundance genes, while allowing true biological outliers to remain high.

The horizontal flat "ceiling" of blue and black dots, present at the top-left corner of the plot, corresponds to low-expression genes with extremely high variance.
Because these genes have extremely low mean counts, they are expected to naturally be filtered out or mathematically penalised during differential expression testing due to their massive variance. They should not skew your downstream results or cause false positives. 

In [ ]:
deseq_dataset <- estimateSizeFactors(deseq_dataset, quiet = TRUE)
deseq_dataset <- estimateDispersions(deseq_dataset, quiet = TRUE)

dispersion_metrics <- data.frame(
    baseMean    = mcols(deseq_dataset)$baseMean,
    dispGeneEst = mcols(deseq_dataset)$dispGeneEst,
    dispersion  = mcols(deseq_dataset)$dispersion,
    dispFit     = mcols(deseq_dataset)$dispFit
)
dispersion_metrics <- dispersion_metrics[dispersion_metrics$baseMean > 0, ]

dispersion_plot <- ggplot(dispersion_metrics, aes(x = baseMean)) +
    geom_point(aes(y = dispGeneEst), color = "black", alpha = 0.3, size = 1) +
    geom_point(aes(y = dispersion), color = "dodgerblue", alpha = 0.3, size = 1, shape = 1) +
    geom_line(aes(y = dispFit), color = "red", linewidth = 1) +
    scale_x_log10() + scale_y_log10() +
    theme_minimal() +
    labs(title = "Dispersion Estimates", x = "Mean of Normalised Counts", y = "Dispersion")

print(dispersion_plot)
ggsave("f_figures/Dispersion_Estimates.png", plot = dispersion_plot, width = 8, height = 6, dpi = 300)

## 4. Dimensionality Reduction & Diversity Testing
Performs Principal Component Analysis (PCA) to check sample clustering, followed by PERMANOVA and Beta Dispersion tests.

**Expected Figure Trends:** The PCA plot is anticipated to display distinct and separate clusters representing the different `LabelingState` and `AssayType` groups along the primary principal components (PC1 and PC2). High biological and technical differentiation will be reflected by tight intra-group sample groupings and large inter-group distances.

In [ ]:
pca_results <- plotPCA(vst_transformed_data, intgroup = c("LabelingState", "AssayType"), ntop = num_pca_genes, returnData = TRUE)

pca_palette <- c("lab:input" = "#D55E00", "lab:pull_down" = "#009E73",
                 "unlab:input" = "#56B4E9", "unlab:pull_down" = "#CC79A7")
pca_shapes <- c("lab:input" = 16, "lab:pull_down" = 17,
                "unlab:input" = 15, "unlab:pull_down" = 18)
pca_labels <- c("lab:input" = "Labeled Input", "lab:pull_down" = "Labeled Pull-Down",
                "unlab:input" = "Unlabeled Input", "unlab:pull_down" = "Unlabeled Pull-Down")

variance_explained <- round(attr(pca_results, "percentVar") * 100, 1)

pca_figure <- ggplot(pca_results, aes(x = PC1, y = PC2, color = group, shape = group)) +
    geom_point(size = 2.5) +
    scale_color_manual(name = "Group", values = pca_palette, labels = pca_labels) +
    scale_shape_manual(name = "Group", values = pca_shapes, labels = pca_labels) +
    labs(x = glue("PC1: {variance_explained[1]}% variance"),
         y = glue("PC2: {variance_explained[2]}% variance")) +
    theme_minimal() +
    theme(panel.border = element_rect(color = "black", fill = NA, linewidth = 0.5))

print(pca_figure)

distance_matrix <- vegdist(t(assay(vst_transformed_data)), method = "euclidean")

print("--- Multi-Factor PERMANOVA (LabelingState * AssayType) ---")
print(adonis2(distance_matrix ~ LabelingState * AssayType, data = experimental_metadata, permutations = 999))

print("--- Single Factor PERMANOVA (AssayType) ---")
print(adonis2(distance_matrix ~ AssayType, data = experimental_metadata, permutations = 999))

print("--- Beta Dispersion ---")
interaction_groups <- interaction(experimental_metadata$LabelingState, experimental_metadata$AssayType)
print(anova(betadisper(distance_matrix, group = interaction_groups)))

## 5. Differential Expression Analysis
Executes the DESeq model, extracts relevant contrasts using LFC shrinkage, and categorises genes into strict confidence tiers.

### Differential Expression Stratification & Tiered Categorisation

To isolate biological signals from technical background noise, the pipeline evaluates three distinct contrasts using DESeq2:

1. **`Lab_PD_vs_Input`**: Evaluates enrichment in the labeled pull-down sample against its paired labeled input background.
2. **`Lab_PD_vs_All_Input`**: Compares the labeled pull-down against a pooled average of both labeled and unlabeled inputs, providing a robust baseline control.
3. **`Lab_vs_Unlab_PD`**: Compares the labeled pull-down directly against the unlabeled pull-down to identify transcripts whose capture is strictly dependent on the presence of the metabolic label.

#### Confidence Tiering Criteria

Post-shrinkage log₂ fold changes ($L2FC$) and significance statistics (adjusted p-values/$qval$ and nominal p-values/$pval$) are combined to group genes into strict confidence tiers based on the directionality and strength of their expression shifts:

Our strong confidence criteria is based on: Subcellular transcriptome sequencing with single cell APEX-seq identifies regulators of cell-cell interactions
Andrew Xue, Bo Cai, Qian Xue, Nianping Liu, Xiaojie Qiu, Rogelio A. Hernández-López, Alice Y. Ting
bioRxiv 2026.03.17.712496; doi: https://doi.org/10.64898/2026.03.17.712496

* **Tier 1 (Strongest Confidence)**: Significant after multiple testing corrections with a substantial fold change.
    * *Up*: $\log_2\text{(Fold Change)} > 0.5$ and $\text{Adjusted p-value} < 0.05$
    * *Down*: $\log_2\text{(Fold Change)} < -0.5$ and $\text{Adjusted p-value} < 0.05$
* **Tier 2 (Moderate Confidence)**: Significant after multiple testing corrections but displaying a milder directional shift.
    * *Up*: $0.0 < \log_2\text{(Fold Change)} \le 0.5$ and $\text{Adjusted p-value} < 0.05$
    * *Down*: $-0.5 \le \log_2\text{(Fold Change)} < 0.0$ and $\text{Adjusted p-value} < 0.05$
* **Tier 3 (Weak Stats, High Magnitude)**: Fails multiple testing corrections but exhibits a sharp change in expression magnitude with a significant unadjusted p-value.
    * *Up*: $\log_2\text{(Fold Change)} > 0.5$ and $\text{Nominal p-value} < 0.05$
    * *Down*: $\log_2\text{(Fold Change)} < -0.5$ and $\text{Nominal p-value} < 0.05$
* **Tier 4 (Weakest Confidence)**: Fails multiple testing corrections and shows a mild directional shift, backed only by a significant unadjusted p-value.
    * *Up*: $0.0 < \log_2\text{(Fold Change)} \le 0.5$ and $\text{Nominal p-value} < 0.05$
    * *Down*: $-0.5 \le \log_2\text{(Fold Change)} < 0.0$ and $\text{Nominal p-value} < 0.05$
* **Not Significant**: Genes failing to meet the minimum threshold criteria ($\text{Nominal p-value} \ge 0.05$).

In [ ]:
deseq_dataset <- DESeq(deseq_dataset, quiet = TRUE)
model_coefficients <- resultsNames(deseq_dataset)

contrast_labeled_pd_vs_input <- list(c("AssayType_pull_down_vs_input", "LabelingStatelab.AssayTypepull_down"))
de_labeled_pd_vs_input <- results(deseq_dataset, contrast = contrast_labeled_pd_vs_input)
shrunk_labeled_pd_vs_input <- lfcShrink(deseq_dataset, contrast = contrast_labeled_pd_vs_input, type = "ashr", quiet = TRUE)

contrast_labeled_pd_vs_all_input <- rep(0, length(model_coefficients))
contrast_labeled_pd_vs_all_input[which(model_coefficients == "AssayType_pull_down_vs_input")] <- 1
contrast_labeled_pd_vs_all_input[which(model_coefficients == "LabelingStatelab.AssayTypepull_down")] <- 0.5
de_labeled_pd_vs_all_input <- results(deseq_dataset, contrast = contrast_labeled_pd_vs_all_input)
shrunk_labeled_pd_vs_all_input <- lfcShrink(deseq_dataset, contrast = contrast_labeled_pd_vs_all_input, type = "ashr", quiet = TRUE)

contrast_labeled_vs_unlabeled_pd <- list(c("LabelingState_lab_vs_unlab", "LabelingStatelab.AssayTypepull_down"))
de_labeled_vs_unlabeled_pd <- results(deseq_dataset, contrast = contrast_labeled_vs_unlabeled_pd)
shrunk_labeled_vs_unlabeled_pd <- lfcShrink(deseq_dataset, contrast = contrast_labeled_vs_unlabeled_pd, type = "ashr", quiet = TRUE)

categorise_genes <- function(unshrunk_data, shrunk_data, fc_cut, padj_cut, pval_cut) {
    results_df <- data.frame(
        log2FoldChange = shrunk_data$log2FoldChange,
        padj = unshrunk_data$padj,
        pvalue = unshrunk_data$pvalue,
        row.names = rownames(shrunk_data)
    )
    results_df <- results_df[!is.na(results_df$padj) & !is.na(results_df$pvalue), ]

    is_highly_upregulated <- results_df$log2FoldChange > fc_cut
    is_moderately_upregulated <- results_df$log2FoldChange > 0 & results_df$log2FoldChange <= fc_cut
    is_highly_downregulated <- results_df$log2FoldChange < -fc_cut
    is_moderately_downregulated <- results_df$log2FoldChange >= -fc_cut & results_df$log2FoldChange < 0

    has_strong_significance <- results_df$padj < padj_cut
    has_weak_significance <- results_df$padj >= padj_cut & results_df$pvalue < pval_cut

    results_df$Confidence_Tier <- "Not Significant"
    results_df$Confidence_Tier[is_highly_upregulated & has_strong_significance]       <- "Tier 1: L2FC>0.5, qval<0.05"
    results_df$Confidence_Tier[is_moderately_upregulated & has_strong_significance]   <- "Tier 2: L2FC>0.0, qval<0.05"
    results_df$Confidence_Tier[is_highly_upregulated & has_weak_significance]         <- "Tier 3: L2FC>0.5, pval<0.05"
    results_df$Confidence_Tier[is_moderately_upregulated & has_weak_significance]     <- "Tier 4: L2FC>0.0, pval<0.05"
    results_df$Confidence_Tier[is_highly_downregulated & has_strong_significance]     <- "Tier 1: L2FC<-0.5, qval<0.05"
    results_df$Confidence_Tier[is_moderately_downregulated & has_strong_significance] <- "Tier 2: L2FC<0.0, qval<0.05"
    results_df$Confidence_Tier[is_highly_downregulated & has_weak_significance]       <- "Tier 3: L2FC<-0.5, pval<0.05"
    results_df$Confidence_Tier[is_moderately_downregulated & has_weak_significance]   <- "Tier 4: L2FC<0.0, pval<0.05"

    all_possible_tiers <- c(
        "Tier 1: L2FC>0.5, qval<0.05",
        "Tier 2: L2FC>0.0, qval<0.05",
        "Tier 3: L2FC>0.5, pval<0.05",
        "Tier 4: L2FC>0.0, pval<0.05",
        "Tier 1: L2FC<-0.5, qval<0.05",
        "Tier 2: L2FC<0.0, qval<0.05",
        "Tier 3: L2FC<-0.5, pval<0.05",
        "Tier 4: L2FC<0.0, pval<0.05",
        "Not Significant"
    )
    
    results_df$Confidence_Tier <- factor(results_df$Confidence_Tier, levels = all_possible_tiers)
    
    return(results_df)
}

de_comparisons <- list(
    "Lab_PD_vs_Input"     = list(unshrunk = de_labeled_pd_vs_input, shrunk = shrunk_labeled_pd_vs_input),
    "Lab_PD_vs_All_Input" = list(unshrunk = de_labeled_pd_vs_all_input, shrunk = shrunk_labeled_pd_vs_all_input),
    "Lab_vs_Unlab_PD"     = list(unshrunk = de_labeled_vs_unlabeled_pd, shrunk = shrunk_labeled_vs_unlabeled_pd)
)

categorised_results <- list()
for (comparison_id in names(de_comparisons)) {
    message(glue("\n--- Categorising Comparison: {comparison_id} ---"))
    tier_df <- categorise_genes(
        de_comparisons[[comparison_id]]$unshrunk,
        de_comparisons[[comparison_id]]$shrunk,
        log2_fc_threshold,
        adjusted_p_threshold,
        nominal_p_threshold
    )
    print(table(tier_df$Confidence_Tier))
    categorised_results[[comparison_id]] <- tier_df
    write.csv(tier_df, file = glue("o_outputs/processed_data/DE_results/Tiered_Results_{comparison_id}.csv"))
}

tier_labeled_pd_vs_input <- categorised_results[["Lab_PD_vs_Input"]]
tier_labeled_pd_vs_all_input <- categorised_results[["Lab_PD_vs_All_Input"]]
tier_labeled_vs_unlabeled_pd <- categorised_results[["Lab_vs_Unlab_PD"]]

## 6. Differential Expression Visualisation (Volcano & MA Plots)

This section maps a custom diagnostic plotting function across all three shrunken DESeq2 contrasts. 

For each comparison, the workflow systematically evaluates log₂ fold change and multiple-testing adjusted p-value boundaries, producing two standard visual outputs:
1. **Volcano Plots**: Dissect biological statistical significance against structural expression magnitude shifts, colour-coding significantly changing transcripts.
2. **MA Plots (Bland-Altman plots)**: Scale the distribution of shrunken log₂ fold changes against the mean of normalised counts to verify that variation balances correctly across low, moderate, and high expression thresholds.

Outputs are printed directly inline and automatically exported using the canonical contrast name identifier keys.

**Expected Figure Trends:**
* **Volcano Plots**: Expected to display a symmetric volcano-like dispersion where significant transcripts breach the defined p-value (horizontal) and log-fold change (vertical) limits, highlighted in red (up-regulated) or blue (down-regulated)
* **MA Plots**: Should confirm stable variance across the range of mean expression, tracking symmetrically around the zero log-fold change axis, demonstrating that significance isn't falsely driven by low-count noise.

In [ ]:
generate_diagnostic_plots <- function(contrast_results, contrast_name, fc_cut, padj_cut, ptype) {
    plot_data <- as.data.frame(contrast_results)
    plot_data <- plot_data[!is.na(plot_data$padj) & !is.na(plot_data$baseMean), ]
    plot_data$Significance <- "Not Significant"
    
    if (ptype == "pval") {
        confidence_label <- "Weak Confidence (Nominal p-value)"
        volcano_y_label  <- "-Log10(Nominal P-value)"
        
        plot_data$Significance[plot_data$log2FoldChange > fc_cut & plot_data$pvalue < padj_cut] <- "Up-regulated"
        plot_data$Significance[plot_data$log2FoldChange < -fc_cut & plot_data$pvalue < padj_cut] <- "Down-regulated"
        volcano_y_metric <- -log10(plot_data$pvalue)
    } else {
        confidence_label <- "High Confidence (Adjusted q-value)"
        volcano_y_label  <- "-Log10(Adjusted P-value)"
        
        plot_data$Significance[plot_data$log2FoldChange > fc_cut & plot_data$padj < padj_cut] <- "Up-regulated"
        plot_data$Significance[plot_data$log2FoldChange < -fc_cut & plot_data$padj < padj_cut] <- "Down-regulated"
        volcano_y_metric <- -log10(plot_data$padj)
    }
    
    volcano_plot <- ggplot(plot_data, aes(x = log2FoldChange, y = volcano_y_metric, color = Significance)) +
        geom_point(alpha = 0.6, size = 1.2) + 
        scale_color_manual(values = c("Up-regulated" = "red", "Down-regulated" = "blue", "Not Significant" = "grey")) +
        geom_vline(xintercept = c(-fc_cut, fc_cut), linetype = "dashed", color = "black") +
        geom_hline(yintercept = -log10(padj_cut), linetype = "dashed", color = "black") +
        theme_minimal() +
        theme(legend.position = "none") + 
        labs(title = "Volcano Plot", x = "Log2 Fold Change", y = volcano_y_label)
    
    ma_plot <- ggplot(plot_data, aes(x = baseMean, y = log2FoldChange, color = Significance)) +
        geom_point(alpha = 0.6, size = 1.2) +
        scale_x_log10(labels = scales::trans_format("log10", scales::math_format(10^.x))) +
        scale_color_manual(values = c("Up-regulated" = "red", "Down-regulated" = "blue", "Not Significant" = "grey")) +
        geom_hline(yintercept = 0, linetype = "solid", color = "black", linewidth = 0.5) +
        geom_hline(yintercept = c(-fc_cut, fc_cut), linetype = "dashed", color = "grey40") +
        coord_cartesian(ylim = c(-4, 4)) +
        theme_minimal() +
        labs(title = "MA Plot", x = "Mean of Normalised Counts", y = "Log2 Fold Change")
    
    combined_panel <- (volcano_plot + ma_plot) + 
        plot_layout(guides = "collect") + 
        plot_annotation(
            title    = glue("Diagnostic Plots: {contrast_name}"),
            subtitle = glue("Confidence Level: {confidence_label}  |  Thresholds: |L2FC| > {fc_cut}, p < {padj_cut}"),
            theme    = theme(
                plot.title    = element_text(size = 14, face = "bold", hjust = 0.5),
                plot.subtitle = element_text(size = 11, face = "italic", color = "grey30", hjust = 0.5)
            )
        )
    
    print(combined_panel)
    ggsave(glue("f_figures/DE/Diagnostic_Panel_{contrast_name}_{ptype}.png"), plot = combined_panel, width = 12, height = 5, dpi = 300)
}

shrunk_contrasts <- list(
    "Lab_PD_vs_Input"     = shrunk_labeled_pd_vs_input,
    "Lab_PD_vs_All_Input" = shrunk_labeled_pd_vs_all_input,
    "Lab_vs_Unlab_PD"     = shrunk_labeled_vs_unlabeled_pd
)

for (contrast_title in names(shrunk_contrasts)) {
    generate_diagnostic_plots(
        contrast_results = shrunk_contrasts[[contrast_title]], 
        contrast_name    = contrast_title, 
        fc_cut           = log2_fc_threshold, 
        padj_cut         = adjusted_p_threshold,
        ptype            = "padj"
    )
}

for (contrast_title in names(shrunk_contrasts)) {
    generate_diagnostic_plots(
        contrast_results = shrunk_contrasts[[contrast_title]], 
        contrast_name    = contrast_title, 
        fc_cut           = log2_fc_threshold, 
        padj_cut         = nominal_p_threshold,
        ptype            = "pval"
    )
}

## 7. Network Analysis (Gene Correlation)
Extracts the top dynamically changing transcripts to build a co-expression network map visualised via `ggplot2`.

**Expected Figure Trends:** The plotted heatmap is expected to highlight clustered expression profiles, identifying subsets of highly correlated genes that share co-expression patterns specifically induced in the labelled pull-down cohort compared to input constraints.

In [ ]:
normalised_counts <- assay(vst_transformed_data)
gene_variances <- apply(normalised_counts, 1, var)
most_variable_genes <- names(sort(gene_variances, decreasing = TRUE)[1:num_network_genes])

heatmap_data <- normalised_counts[most_variable_genes, ] %>%
    as.data.frame() %>%
    tibble::rownames_to_column(var = "Gene") %>%
    pivot_longer(cols = -Gene, names_to = "Sample", values_to = "Expression")
experimental_metadata_long <- experimental_metadata %>%
    tibble::rownames_to_column(var = "Sample")
heatmap_data <- heatmap_data %>% left_join(experimental_metadata_long, by = "Sample")
natural_order <- gtools::mixedsort(unique(heatmap_data$Sample))
heatmap_data$Sample <- factor(heatmap_data$Sample, levels = natural_order)

gg_heatmap <- ggplot(heatmap_data, aes(x = Sample, y = Gene, fill = Expression)) +
    geom_tile() +
    scale_fill_viridis_c(option = "magma", name = "VST Counts") + 
    labs(
        title = glue("Co-expression of Top {num_network_genes} Variable Genes"),
        x = "Samples",
        y = "Genes"
    ) +
    theme_minimal() +
    theme(
        axis.text.y = element_blank(),
        axis.text.x = element_text(angle = 45, hjust = 1),
        panel.grid = element_blank()
    )

print(gg_heatmap)

ggsave(
    filename = glue("f_figures/Network-coexpr/Top_{num_network_genes}_Coexpression_Heatmap.png"),
    plot = gg_heatmap,
    width = 8, height = 6, dpi = 300
)

## 8. Gene Ontology (GO) Enrichment Analysis & Visualisation
Identifies enriched biological processes among target gene clusters and produces dotplot visualisations.

**Expected Figure Trends:** The generated dotplots will map enriched functional categories on the y-axis against the gene ratio on the x-axis. Larger dots represent higher target gene hit counts per pathway, with colour intensity signifying stronger adjusted p-values. It is anticipated that ciliary, metabolic, or translation-related terms will emerge in the strongly upregulated clusters.

In [ ]:
extract_genes_by_tier <- function(dataset, valid_tiers) {
    rownames(dataset[dataset$Confidence_Tier %in% valid_tiers, ])
}

execute_go_enrichment <- function(target_genes, background_universe) {
    if (length(target_genes) == 0) return(NULL)
    enrichGO(gene = target_genes, universe = background_universe, OrgDb = org.Mm.eg.db,
             keyType = "SYMBOL", ont = "ALL", pAdjustMethod = "BH",
             pvalueCutoff = 0.05, qvalueCutoff = 0.2, readable = TRUE)
}

assess_cilia_terms <- function(go_results_df, analysis_label) {
    if (nrow(go_results_df) == 0) return(message(glue("No enriched GO terms for {analysis_label}.")))
    cilia_matches <- go_results_df[grep("cili", go_results_df$Description, ignore.case = TRUE), ]
    message(glue("Cilium-related terms in {analysis_label}: {nrow(cilia_matches)}"))
    if (nrow(cilia_matches) > 0) {
        unique_genes <- sort(unique(unlist(strsplit(paste(cilia_matches$geneID, collapse = "/"), "/"))))
        message(glue("Found {length(unique_genes)} unique cilia genes. Sample: "), paste(head(unique_genes), collapse=", "))
    }
}

generate_go_dotplot <- function(go_object, plot_title, output_filename) {
    if (is.null(go_object) || nrow(go_object) == 0) return()
    dotplot_figure <- dotplot(go_object, showCategory = 20) +
        ggtitle(plot_title) +
        theme(axis.text.y = element_text(size = 10), axis.title.y = element_blank())
    print(dotplot_figure)
    ggsave(output_filename, plot = dotplot_figure, width = 10, height = 8, dpi = 300)
}

tiered_datasets <- list(
    "Lab_PD_vs_Input"     = tier_labeled_pd_vs_input,
    "Lab_PD_vs_All_Input" = tier_labeled_pd_vs_all_input,
    "Lab_vs_Unlab_PD"     = tier_labeled_vs_unlabeled_pd
)

for (comparison_name in names(tiered_datasets)) {
    
    message(glue("\n=================================================="))
    message(glue("Running GO Enrichment for Comparison: {comparison_name}"))
    message(glue("=================================================="))
    
    active_dataset <- tiered_datasets[[comparison_name]]
    gene_universe  <- rownames(active_dataset)
    
    genes_strong_up   <- extract_genes_by_tier(active_dataset, c("Tier 1: L2FC>0.5, qval<0.05", "Tier 2: L2FC>0.0, qval<0.05"))
    genes_strong_down <- extract_genes_by_tier(active_dataset, c("Tier 1: L2FC<-0.5, qval<0.05", "Tier 2: L2FC<0.0, qval<0.05"))
    genes_weak_up   <- extract_genes_by_tier(active_dataset, c("Tier 3: L2FC>0.5, pval<0.05", "Tier 4: L2FC>0.0, pval<0.05"))
    genes_weak_down <- extract_genes_by_tier(active_dataset, c("Tier 3: L2FC<-0.5, pval<0.05", "Tier 4: L2FC<0.0, pval<0.05"))
    
    go_res_strong_up   <- execute_go_enrichment(genes_strong_up, gene_universe)
    go_res_strong_down <- execute_go_enrichment(genes_strong_down, gene_universe)
    go_res_weak_up   <- execute_go_enrichment(genes_weak_up, gene_universe)
    go_res_weak_down <- execute_go_enrichment(genes_weak_down, gene_universe)
    
    if (!is.null(go_res_strong_up))   write.csv(as.data.frame(go_res_strong_up),   glue("o_outputs/processed_data/GO_results/{comparison_name}_GO_StrongConf_UP.csv"))
    if (!is.null(go_res_strong_down)) write.csv(as.data.frame(go_res_strong_down), glue("o_outputs/processed_data/GO_results/{comparison_name}_GO_StrongConf_DOWN.csv"))
    if (!is.null(go_res_weak_up))   write.csv(as.data.frame(go_res_weak_up),   glue("o_outputs/processed_data/GO_results/{comparison_name}_GO_WeakConf_UP.csv"))
    if (!is.null(go_res_weak_down)) write.csv(as.data.frame(go_res_weak_down), glue("o_outputs/processed_data/GO_results/{comparison_name}_GO_WeakConf_DOWN.csv"))
    
    assess_cilia_terms(if (!is.null(go_res_strong_up)) as.data.frame(go_res_strong_up) else data.frame(), glue("{comparison_name} (Strong-Confidence Up Targets)"))
    generate_go_dotplot(go_res_strong_up,   glue("Top Processes: Strong-Conf Up-regulated in {comparison_name}"),   glue("f_figures/GO/{comparison_name}_GO_StrongConf_UP.png"))
    generate_go_dotplot(go_res_strong_down, glue("Top Processes: Strong-Conf Down-regulated in {comparison_name}"), glue("f_figures/GO/{comparison_name}_GO_StrongConf_DOWN.png"))
    
    assess_cilia_terms(if (!is.null(go_res_weak_up)) as.data.frame(go_res_weak_up) else data.frame(), glue("{comparison_name} (Weak-Confidence Up Targets)"))
    generate_go_dotplot(go_res_weak_up,   glue("Top Processes: Weak-Conf Up-regulated in {comparison_name}"),   glue("f_figures/GO/{comparison_name}_GO_WeakConf_UP.png"))
    generate_go_dotplot(go_res_weak_down, glue("Top Processes: Weak-Conf Down-regulated in {comparison_name}"), glue("f_figures/GO/{comparison_name}_GO_WeakConf_DOWN.png"))
}

## 9. Gene Set Enrichment Analysis (GSEA)
Generates a ranked metric for genes to evaluate broader pathway activation/suppression, outputting global visualisations and targeted running scores.

**Expected Figure Trends:** The GSEA dotplot should partition overall biological themes structurally by direction of enrichment. The targeted mountain plot is expected to chart a pronounced running enrichment score that peaks towards the top or bottom extreme of the ranked gene list, indicating coordinate induction or suppression of the entire ciliary pathway module.

In [ ]:
shrunk_contrasts <- list(
    "Lab_PD_vs_Input"     = shrunk_labeled_pd_vs_input,
    "Lab_PD_vs_All_Input" = shrunk_labeled_pd_vs_all_input,
    "Lab_vs_Unlab_PD"     = shrunk_labeled_vs_unlabeled_pd
)

for (comparison_name in names(shrunk_contrasts)) {
    
    message(glue("\n=================================================="))
    message(glue("Running GSEA Enrichment for Comparison: {comparison_name}"))
    message(glue("=================================================="))
    
    contrast_results <- shrunk_contrasts[[comparison_name]]
    gsea_data <- as.data.frame(contrast_results)
    gsea_data <- gsea_data[!is.na(gsea_data$pvalue) & !is.na(gsea_data$log2FoldChange), ]
    
    # Generate ranked metric based on Wald stat approximation (LFC / SE)
    gene_ranking_metric <- gsea_data$log2FoldChange / gsea_data$lfcSE
    names(gene_ranking_metric) <- rownames(gsea_data)
    gene_ranking_metric <- sort(gene_ranking_metric[!is.na(gene_ranking_metric)], decreasing = TRUE)
    
    gsea_results_obj <- gseGO(geneList = gene_ranking_metric, OrgDb = org.Mm.eg.db, keyType = "SYMBOL",
                              ont = "ALL", pvalueCutoff = 0.1, pAdjustMethod = "BH", eps = 0, seed = 123)
    
    if (!is.null(gsea_results_obj) && nrow(gsea_results_obj) > 0) {
        gsea_df <- as.data.frame(gsea_results_obj)
        write.csv(gsea_df, glue("o_outputs/processed_data/GO_results/{comparison_name}_GSEA_GO_Enrichment.csv"))
        
        gsea_cilia_terms <- gsea_df[grep("cili", gsea_df$Description, ignore.case = TRUE), ]
        message(glue("GSEA Cilia pathways identified for {comparison_name}: {nrow(gsea_cilia_terms)}"))
        
        # 1. Generate Global GSEA Dotplot Profiles
        gsea_dotplot <- dotplot(gsea_results_obj, showCategory = 15, split = ".sign") +
            facet_grid(. ~ .sign) +
            ggtitle(glue("GSEA Functional Shifts: {comparison_name}")) +
            theme(axis.text.y = element_text(size = 8))
        
        print(gsea_dotplot)
        ggsave(glue("f_figures/GO/{comparison_name}_GSEA_Dotplot.png"), plot = gsea_dotplot, width = 11, height = 8, dpi = 300)
        
        # 2. Generate Target Pathway GSEA Running Score Mountain Plots
        if (nrow(gsea_cilia_terms) > 0) {
            gsea_mountain_plot <- gseaplot2(gsea_results_obj, geneSetID = gsea_cilia_terms$ID[1],
                                            title = glue("{comparison_name} Running Enrichment: {gsea_cilia_terms$Description[1]}"))
            print(gsea_mountain_plot)
            ggsave(glue("f_figures/GO/{comparison_name}_GSEA_Mountain_Plot_Cilia.png"), plot = gsea_mountain_plot, width = 8, height = 6, dpi = 300)
        }
    } else {
        message(glue("No significant GSEA pathways enriched for comparison: {comparison_name}"))
    }
}

## 10. Deep-Dive: Ciliary Basal Body Bi-directional Drivers
Extracts "leading edge" genes from the GSEA mountain plot to identify specific drivers of bi-directional pathway shifts. Visualises expression, builds isolated co-expression networks, and exports lists for motif analysis.

**Expected Figure Trends:** Replacing previous base-R implementations with `ggplot2` representations, the resulting heatmaps will explicitly map core sub-network components. The deep-dive expression map will stratify condition-dependent gene transcription variations, whereas the correlation heatmap will outline robustly co-regulated gene modules mapping the functional landscape of the queried biological pathways.

In [ ]:
shrunk_contrasts <- list(
    "Lab_PD_vs_Input"     = shrunk_labeled_pd_vs_input,
    "Lab_PD_vs_All_Input" = shrunk_labeled_pd_vs_all_input,
    "Lab_vs_Unlab_PD"     = shrunk_labeled_vs_unlabeled_pd
)

for (comparison_name in names(shrunk_contrasts)) {
    
    # 1. Load the specific GSEA outputs calculated in the previous section
    gsea_file_path <- glue("o_outputs/processed_data/GO_results/{comparison_name}_GSEA_GO_Enrichment.csv")
    
    if (!file.exists(gsea_file_path)) {
        message(glue("\nSkipping {comparison_name}: No GSEA data file found."))
        next
    }
    
    gsea_df <- read.csv(gsea_file_path, stringsAsFactors = FALSE)
    gsea_cilia_terms <- gsea_df[grep("cili", gsea_df$Description, ignore.case = TRUE), ]
    
    if (nrow(gsea_cilia_terms) == 0) {
        message(glue("\nSkipping {comparison_name}: No cilium-related pathways found in GSEA results."))
        next
    }

    primary_term_id <- gsea_cilia_terms$ID[1]
    primary_term_desc <- gsea_cilia_terms$Description[1]
    sanitised_term_name <- gsub(" ", "_", primary_term_desc)

    message(glue("\n=================================================="))
    message(glue("Deep-Dive Drivers [{comparison_name}]: {primary_term_desc} ({primary_term_id})"))
    message(glue("=================================================="))

    leading_edge_string <- gsea_cilia_terms$core_enrichment[1]
    leading_edge_genes <- unlist(strsplit(leading_edge_string, "/"))

    # Isolate contrast-specific LFC profiles to sort driver vectors
    contrast_data <- as.data.frame(shrunk_contrasts[[comparison_name]])
    gene_lfc_values <- contrast_data[leading_edge_genes, "log2FoldChange"]
    names(gene_lfc_values) <- leading_edge_genes

    upregulated_drivers <- names(gene_lfc_values[!is.na(gene_lfc_values) & gene_lfc_values > 0])
    downregulated_drivers <- names(gene_lfc_values[!is.na(gene_lfc_values) & gene_lfc_values < 0])

    message(glue("Total Leading Edge Genes: {length(leading_edge_genes)}"))
    message(glue("Up-regulated Drivers: {length(upregulated_drivers)}"))
    message(glue("Down-regulated Drivers: {length(downregulated_drivers)}"))

    write.table(upregulated_drivers, glue("o_outputs/processed_data/GO_results/{comparison_name}_{sanitised_term_name}_Up_Drivers.txt"),
                row.names = FALSE, col.names = FALSE, quote = FALSE)
    write.table(downregulated_drivers, glue("o_outputs/processed_data/GO_results/{comparison_name}_{sanitised_term_name}_Down_Drivers.txt"),
                row.names = FALSE, col.names = FALSE, quote = FALSE)

    core_driver_counts <- normalised_counts[intersect(leading_edge_genes, rownames(normalised_counts)), ]

    # 2. Shape driver counts data into a tidy format for ggplot2
    driver_expr_df <- as.data.frame(core_driver_counts) %>%
        tibble::rownames_to_column(var = "Gene") %>%
        pivot_longer(cols = -Gene, names_to = "Sample", values_to = "Expression")

    # 3. Join with metadata
    experimental_metadata_long <- experimental_metadata %>%
        tibble::rownames_to_column(var = "Sample")
    driver_expr_df <- driver_expr_df %>% left_join(experimental_metadata_long, by = "Sample")

    # 4. Force natural numerical factor sorting AFTER the join transformation
    natural_sample_order <- gtools::mixedsort(unique(driver_expr_df$Sample))
    driver_expr_df$Sample <- factor(driver_expr_df$Sample, levels = natural_sample_order)

    # 5. Generate and export the contrast-specific Expression Heatmap
    expr_heatmap <- ggplot(driver_expr_df, aes(x = Sample, y = Gene, fill = Expression)) +
        geom_tile(color = "white") +
        scale_fill_viridis_c(option = "magma", name = "Normalised Counts") +
        theme_minimal() +
        labs(title = glue("{comparison_name} - Driver Expression: {primary_term_desc}"), x = "Sample", y = "Gene") +
        theme(axis.text.x = element_text(angle = 45, hjust = 1), axis.text.y = element_text(size = 6))

    print(expr_heatmap)
    flush.console()
    ggsave(glue("f_figures/GO/{comparison_name}_{sanitised_term_name}_Expression_Heatmap.png"), plot = expr_heatmap, width = 8, height = 10, dpi = 300)

    # 6. Generate and export the contrast-specific Co-expression Network Correlation Heatmap
    core_driver_correlation <- cor(t(core_driver_counts), method = "pearson")
    
    driver_cor_df <- as.data.frame(core_driver_correlation) %>%
        tibble::rownames_to_column(var = "Gene1") %>%
        pivot_longer(cols = -Gene1, names_to = "Gene2", values_to = "Correlation")

    cor_heatmap <- ggplot(driver_cor_df, aes(x = Gene1, y = Gene2, fill = Correlation)) +
        geom_tile(color = "white") +
        scale_fill_gradient2(low = "blue", high = "red", mid = "white", midpoint = 0, limit = c(-1, 1), name = "Pearson r") +
        theme_minimal() +
        labs(title = glue("{comparison_name} - Co-expression Network: {primary_term_desc}"), x = "Gene", y = "Gene") +
        theme(axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 6), axis.text.y = element_text(size = 6))

    print(cor_heatmap)
    flush.console()
    ggsave(glue("f_figures/Network-coexpr/{comparison_name}_{sanitised_term_name}_Coexpression_Heatmap.png"), plot = cor_heatmap, width = 10, height = 10, dpi = 300)

    message(glue("Analysis complete for {comparison_name}. Plots and driver sub-selections saved successfully."))
}